# Tech Challenge Fase 3 - State of Data Brasil
## Notebook de Modelagem: Silver -> Gold

Le as 2 tabelas Silver (`perfil_profissional` - uma linha por respondente - e `respostas_multipla_escolha` - uma linha por marcacao em pergunta de multipla escolha) e constroi as 10 tabelas Gold, uma por recorte de negocio, gravando cada uma em Parquet particionado por `ano_pesquisa` no S3.

**Tabelas Gold geradas (10):** `gold_estrutura_mercado`, `gold_perfis_profissionais`, `gold_diversidade`, `gold_diversidade_cargo`, `gold_adocao_tecnologia`, `gold_adocao_ia_prioridade`, `gold_adocao_ia_motivos`, `gold_regiao_senioridade_modelo`, `gold_retencao`, `gold_desafios_gestor`.

### Setup: sessao interativa

In [1]:
%idle_timeout 30
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import functions as F

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

BUCKET = "techchallenge-fase3-fiap-761328510173"
SILVER_WIDE_PATH = f"s3://{BUCKET}/silver/perfil_profissional/"
SILVER_LONG_PATH = f"s3://{BUCKET}/silver/respostas_multipla_escolha/"
GOLD_BASE_PATH = f"s3://{BUCKET}/gold/"

print("Setup concluido.")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 5.1
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 2
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 2
Idle Timeout: 30
Session ID: f61b4c19-b6e4-4b83-84be-0265564f6937
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session f61b4c19-b6e4-4b83-84be-0265564f6937 to get into ready status...
Session f61b4c19-b6e4-4b83-84be-0265564f6937 has 

### Leitura das 2 tabelas Silver

In [2]:
wide = spark.read.parquet(SILVER_WIDE_PATH)
long = spark.read.parquet(SILVER_LONG_PATH)

print("perfil_profissional:", wide.count(), "linhas")
print("respostas_multipla_escolha:", long.count(), "linhas")

perfil_profissional: 14002 linhas
respostas_multipla_escolha: 62483 linhas


### Pergunta 1 - gold_estrutura_mercado
Como esta estruturado o mercado brasileiro de Dados: setor de atuacao, porte de empresa e tamanho do time de dados.

In [3]:
gold_estrutura_mercado = (
    wide.groupBy("ano_pesquisa", "setor", "num_funcionarios", "num_pessoas_dados")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)
gold_estrutura_mercado.show(5)

+------------+--------------------+----------------+-----------------+------------------+
|ano_pesquisa|               setor|num_funcionarios|num_pessoas_dados|total_respondentes|
+------------+--------------------+----------------+-----------------+------------------+
|        2025|           Indústria|     de 51 a 100|          11 - 20|                 1|
|        2025|    Setor Automotivo|  Acima de 3.000|    Não informado|                26|
|        2025|           Indústria|  Acima de 3.000|    Não informado|                87|
|        2025|Seguros ou Previd...|    de 101 a 500|    Não informado|                 9|
|        2025|       Área da Saúde|  Acima de 3.000|    Não informado|                37|
+------------+--------------------+----------------+-----------------+------------------+
only showing top 5 rows


### Pergunta 2 - gold_perfis_profissionais
Cargo mais comum e progressao salarial por senioridade.

In [5]:
gold_perfis_profissionais = (
    wide.groupBy("ano_pesquisa", "cargo", "senioridade", "faixa_salarial")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)
gold_perfis_profissionais.show(5)

+------------+--------------------+-------------------+--------------------+------------------+
|ano_pesquisa|               cargo|        senioridade|      faixa_salarial|total_respondentes|
+------------+--------------------+-------------------+--------------------+------------------+
|        2025|       Não aplicável|      Não aplicável|de R$ 30.001/mês ...|                56|
|        2025|Outras Engenharia...|              Pleno|de R$ 12.001/mês ...|                 4|
|        2025|Cientista de Dado...|              Pleno|de R$ 8.001/mês a...|                67|
|        2025|Analista de Dados...|             Júnior|de R$ 3.001/mês a...|                28|
|        2025|Engenheiro de Dad...|Especialista/Staff+|de R$ 16.001/mês ...|                18|
+------------+--------------------+-------------------+--------------------+------------------+
only showing top 5 rows


### Pergunta 3 - gold_diversidade e gold_diversidade_cargo
Diversidade de genero, raca/etnia e PCD, cruzada com senioridade e faixa salarial. A segunda tabela cruza cargo x genero, para responder a representatividade feminina por cargo.

In [4]:
gold_diversidade = (
    wide.groupBy("ano_pesquisa", "genero", "raca_etnia", "pcd", "senioridade", "faixa_salarial")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)
gold_diversidade.show(5)

gold_diversidade_cargo = (
    wide.groupBy("ano_pesquisa", "cargo", "genero")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)
gold_diversidade_cargo.show(5)

+------------+---------+----------+---+-------------+--------------------+------------------+
|ano_pesquisa|   genero|raca_etnia|pcd|  senioridade|      faixa_salarial|total_respondentes|
+------------+---------+----------+---+-------------+--------------------+------------------+
|        2025|Masculino|    Branca|Não|Não aplicável|de R$ 30.001/mês ...|                36|
|        2025|Masculino|    Branca|Não|       Sênior|de R$ 25.001/mês ...|                19|
|        2025| Feminino|     Parda|Não|       Júnior|de R$ 3.001/mês a...|                 7|
|        2025|Masculino|    Branca|Não|        Pleno|de R$ 8.001/mês a...|               119|
|        2025|Masculino|    Branca|Não|       Sênior|de R$ 8.001/mês a...|               127|
+------------+---------+----------+---+-------------+--------------------+------------------+
only showing top 5 rows

+------------+--------------------+---------+------------------+
|ano_pesquisa|               cargo|   genero|total_respondentes|

### Pergunta 4 - gold_adocao_tecnologia
Tres categorias na mesma tabela: `linguagem_preferida` e `ferramenta_bi_preferida` vem direto da Silver "wide" (pergunta de resposta unica); `cloud_usada` e `banco_dados_usado` vem da Silver "long" (multipla escolha, ja em formato explodido).

In [6]:
ling = (
    wide.groupBy("ano_pesquisa", "linguagem_preferida")
    .count()
    .withColumnRenamed("count", "total_respondentes")
    .withColumnRenamed("linguagem_preferida", "item")
    .withColumn("categoria", F.lit("linguagem_preferida"))
)

bi_pref = (
    wide.groupBy("ano_pesquisa", "bi_preferida")
    .count()
    .withColumnRenamed("count", "total_respondentes")
    .withColumnRenamed("bi_preferida", "item")
    .withColumn("categoria", F.lit("ferramenta_bi_preferida"))
)

tech_long = long.filter(F.col("bloco").isin("cloud_usada", "banco_dados_usado"))
tech_agg = (
    tech_long.groupBy("ano_pesquisa", "bloco", "item")
    .count()
    .withColumnRenamed("count", "total_respondentes")
    .withColumnRenamed("bloco", "categoria")
)

gold_adocao_tecnologia = (
    ling.select("ano_pesquisa", "categoria", "item", "total_respondentes")
    .unionByName(bi_pref.select("ano_pesquisa", "categoria", "item", "total_respondentes"))
    .unionByName(tech_agg.select("ano_pesquisa", "categoria", "item", "total_respondentes"))
)

gold_adocao_tecnologia.show(5)
gold_adocao_tecnologia.filter(F.col("categoria") == "ferramenta_bi_preferida").show(5, truncate=False)

+------------+-------------------+--------------------+------------------+
|ano_pesquisa|          categoria|                item|total_respondentes|
+------------+-------------------+--------------------+------------------+
|        2025|linguagem_preferida|              Python|               254|
|        2025|linguagem_preferida|         SQL, Python|               506|
|        2025|linguagem_preferida|    Python, Julia, R|                 1|
|        2025|linguagem_preferida|    SQL, Python, DAX|                 2|
|        2025|linguagem_preferida|C/C++/C#, SQL, Py...|                 1|
+------------+-------------------+--------------------+------------------+
only showing top 5 rows

+------------+-----------------------+--------------------------------------+------------------+
|ano_pesquisa|categoria              |item                                  |total_respondentes|
+------------+-----------------------+--------------------------------------+------------------+
|2025    

### Pergunta 5 - gold_adocao_ia_prioridade e gold_adocao_ia_motivos
Uso individual de IA generativa e prioridade estrategica declarada pela empresa; principais motivos relatados para nao adotar IA generativa.

In [7]:
gold_adocao_ia_prioridade = (
    wide.groupBy("ano_pesquisa", "ia_prioridade", "usa_ia_generativa")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)

gold_adocao_ia_motivos = (
    long.filter(F.col("bloco") == "motivo_nao_ia")
    .groupBy("ano_pesquisa", "item")
    .count()
    .withColumnRenamed("count", "total_mencoes")
)

gold_adocao_ia_prioridade.show(5)
gold_adocao_ia_motivos.orderBy(F.desc("total_mencoes")).show(5)

+------------+--------------------+--------------------+------------------+
|ano_pesquisa|       ia_prioridade|   usa_ia_generativa|total_respondentes|
+------------+--------------------+--------------------+------------------+
|        2025|       Não informado|Utilizo apenas so...|                75|
|        2025|Mais ou menos... ...|       Não informado|               169|
|        2025|       Não informado|Utilizo soluções ...|               224|
|        2025|       Não informado|Utilizo apenas so...|               460|
|        2025|       Não informado|Utilizo soluções ...|               122|
+------------+--------------------+--------------------+------------------+
only showing top 5 rows

+------------+--------------------+-------------+
|ano_pesquisa|                item|total_mencoes|
+------------+--------------------+-------------+
|        2024|Falta de compreen...|          334|
|        2024|Falta de expertis...|          328|
|        2024|Preocupações com ...|      

### Pergunta 6 - gold_regiao_senioridade_modelo
Regiao, senioridade, modelo de trabalho e faixa salarial - permite cruzar qualquer combinacao dessas 4 dimensoes.

In [8]:
gold_regiao_senioridade_modelo = (
    wide.groupBy("ano_pesquisa", "regiao", "senioridade", "modelo_trabalho", "faixa_salarial")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)
gold_regiao_senioridade_modelo.show(5)

+------------+-------+-------------+--------------------+--------------------+------------------+
|ano_pesquisa| regiao|  senioridade|     modelo_trabalho|      faixa_salarial|total_respondentes|
+------------+-------+-------------+--------------------+--------------------+------------------+
|        2025|Sudeste|       Sênior|  Modelo 100% remoto|de R$ 25.001/mês ...|                 8|
|        2025|Sudeste|Não aplicável|Modelo híbrido co...|de R$ 6.001/mês a...|                 1|
|        2025|    Sul|       Sênior|Modelo híbrido fl...|de R$ 6.001/mês a...|                 2|
|        2025|Sudeste|Não aplicável|  Modelo 100% remoto|de R$ 25.001/mês ...|                21|
|        2025|Sudeste|        Pleno|  Modelo 100% remoto|de R$ 8.001/mês a...|                59|
+------------+-------+-------------+--------------------+--------------------+------------------+
only showing top 5 rows


### Pergunta 7 - gold_retencao e gold_desafios_gestor
Satisfacao atual cruzada com intencao de mudar de emprego (retencao); principais desafios relatados por gestores de dados.

In [9]:
gold_retencao = (
    wide.groupBy("ano_pesquisa", "satisfeito_atualmente", "planeja_mudar_6m")
    .count()
    .withColumnRenamed("count", "total_respondentes")
)

gold_desafios_gestor = (
    long.filter(F.col("bloco") == "desafio_gestor")
    .groupBy("ano_pesquisa", "item")
    .count()
    .withColumnRenamed("count", "total_mencoes")
)

gold_retencao.show(5)
gold_desafios_gestor.orderBy(F.desc("total_mencoes")).show(5)

+------------+---------------------+--------------------+------------------+
|ano_pesquisa|satisfeito_atualmente|    planeja_mudar_6m|total_respondentes|
+------------+---------------------+--------------------+------------------+
|        2025|                  Sim|Não estou buscand...|              1031|
|        2025|                  Sim|Estou em busca de...|               189|
|        2025|                  Não|Não estou buscand...|                44|
|        2025|                  Não|Estou em busca de...|               572|
|        2025|                  Não|Não estou buscand...|               275|
+------------+---------------------+--------------------+------------------+
only showing top 5 rows

+------------+--------------------+-------------+
|ano_pesquisa|                item|total_mencoes|
+------------+--------------------+-------------+
|        2024|Gerenciar a expec...|          367|
|        2024|Dividir o tempo e...|          319|
|        2023|Gerenciar a expec.

### Validacao antes de gravar
As tabelas construidas a partir de uma linha por respondente (sem explodir multipla escolha) devem somar exatamente 14.002 - o total de respondentes validos apos a deduplicacao na Silver. Confere isso antes de gravar qualquer coisa.

In [10]:
for nome, df in [
    ("gold_estrutura_mercado", gold_estrutura_mercado),
    ("gold_perfis_profissionais", gold_perfis_profissionais),
    ("gold_diversidade", gold_diversidade),
    ("gold_diversidade_cargo", gold_diversidade_cargo),
    ("gold_regiao_senioridade_modelo", gold_regiao_senioridade_modelo),
    ("gold_retencao", gold_retencao),
]:
    total = df.agg(F.sum("total_respondentes")).collect()[0][0]
    status = "OK" if total == 14002 else "DIVERGENTE - INVESTIGAR ANTES DE GRAVAR"
    print(f"{nome}: soma = {total} -- {status}")

gold_estrutura_mercado: soma = 14002 -- OK
gold_perfis_profissionais: soma = 14002 -- OK
gold_diversidade: soma = 14002 -- OK
gold_diversidade_cargo: soma = 14002 -- OK
gold_regiao_senioridade_modelo: soma = 14002 -- OK
gold_retencao: soma = 14002 -- OK


### Gravacao final das 10 tabelas Gold
So roda depois que a validacao acima mostrar "OK" em todas as linhas.

In [11]:
tabelas_gold = {
    "gold_estrutura_mercado": gold_estrutura_mercado,
    "gold_perfis_profissionais": gold_perfis_profissionais,
    "gold_diversidade": gold_diversidade,
    "gold_diversidade_cargo": gold_diversidade_cargo,
    "gold_adocao_tecnologia": gold_adocao_tecnologia,
    "gold_adocao_ia_prioridade": gold_adocao_ia_prioridade,
    "gold_adocao_ia_motivos": gold_adocao_ia_motivos,
    "gold_regiao_senioridade_modelo": gold_regiao_senioridade_modelo,
    "gold_retencao": gold_retencao,
    "gold_desafios_gestor": gold_desafios_gestor,
}

for nome, df in tabelas_gold.items():
    caminho = f"{GOLD_BASE_PATH}{nome}/"
    df.write.mode("overwrite").partitionBy("ano_pesquisa").parquet(caminho)
    print(f"Gravado: {nome}")

print("\nTodas as 10 tabelas Gold gravadas com sucesso.")

Gravado: gold_estrutura_mercado
Gravado: gold_perfis_profissionais
Gravado: gold_diversidade
Gravado: gold_diversidade_cargo
Gravado: gold_adocao_tecnologia
Gravado: gold_adocao_ia_prioridade
Gravado: gold_adocao_ia_motivos
Gravado: gold_regiao_senioridade_modelo
Gravado: gold_retencao
Gravado: gold_desafios_gestor

Todas as 10 tabelas Gold gravadas com sucesso.


### Encerramento da sessao
Rode a celula abaixo (ou clique em **"Pare o caderno"** no Console) ao terminar, para evitar cobranca continua de DPU-hora.

In [ ]:
%stop_session